# BGE cross-encoder fine-tuning and scoring (Colab)

Model and training objective match the Kaggle BGE notebook: `BAAI/bge-reranker-v2-m3`, full sequence-classifier fine-tuning, the same train/dev split, up to 1,000,000 train pairs, one pass with the same optimizer/loss, and tune/holdout/test scoring. Colab-specific changes are the Drive paths and gradient accumulation for its single GPU.

1. Upload the four `ce_bundle_big` parquet files into a private Google Drive folder named `ber_ce_big`.
2. Select **Runtime → Change runtime type → T4 GPU**.
3. Run all cells. Outputs are written to `My Drive/ber_ce_big/out/`.

Training columns: `ta`, `tb`, `label`. Scoring columns: `lid`, `rid`, `ta`, `tb`.

In [ ]:
!pip -q install -U "transformers<5" pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')
import os, time, math, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BUNDLE = Path('/content/drive/MyDrive/ber_ce_big')
OUT = BUNDLE / 'out'
OUT.mkdir(parents=True, exist_ok=True)
expected = ['train.parquet', 'score_tune.parquet', 'score_holdout.parquet', 'score_test.parquet']
missing = [name for name in expected if not (BUNDLE / name).exists()]
assert not missing, f'Missing {missing} under {BUNDLE}; upload all four parquet files to My Drive/ber_ce_big.'
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU.'
print(BUNDLE, os.listdir(BUNDLE), torch.cuda.get_device_name(0), flush=True)

BASE, MAXLEN, SEED = 'BAAI/bge-reranker-v2-m3', 80, 42
MAX_TRAIN, MICRO_BS, GRAD_ACCUM, LR, TRAIN_MINUTES = 1_000_000, 8, 12, 2e-5, 170
# 8 examples fit a single T4; accumulation preserves Kaggle's effective batch size of 96.
torch.manual_seed(SEED); np.random.seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1).cuda()
model.config.pad_token_id = tok.pad_token_id
tr = pd.read_parquet(BUNDLE / 'train.parquet').sample(frac=1.0, random_state=SEED).reset_index(drop=True)
dev, tr = tr.iloc[:10000].reset_index(drop=True), tr.iloc[10000:10000 + MAX_TRAIN].reset_index(drop=True)
print('train', len(tr), 'pos rate', round(float(tr.label.mean()), 4), 'dev', len(dev), flush=True)


In [ ]:
def batches(df, bs, shuffle, seed=0):
    idx = np.random.default_rng(seed).permutation(len(df)) if shuffle else np.arange(len(df))
    for s in range(0, len(df), bs):
        j = idx[s:s + bs]
        enc = tok(df.ta.values[j].tolist(), df.tb.values[j].tolist(), truncation=True, max_length=MAXLEN,
                  padding=True, return_tensors='pt')
        yield j, {k: v.cuda(non_blocking=True) for k, v in enc.items()}

def evaluate(df, bs=32):
    model.eval(); out = np.zeros(len(df), dtype=np.float32)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for j, enc in batches(df, bs, False):
            out[j] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    model.train(); return out

steps = math.ceil(len(tr) / (MICRO_BS * GRAD_ACCUM))
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 500) * max(0.0, 1 - s / steps))
scaler = torch.amp.GradScaler('cuda')
y_all = torch.tensor(tr.label.values, dtype=torch.float32)
t0 = time.time(); model.train(); opt.zero_grad(set_to_none=True)
update = 0; accum = 0
for micro, (j, enc) in enumerate(batches(tr, MICRO_BS, True, SEED), 1):
    with torch.autocast('cuda', dtype=torch.float16):
        logit = model(**enc).logits.squeeze(-1)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logit.float(), y_all[j].cuda())
    scaler.scale(loss / GRAD_ACCUM).backward(); accum += 1
    if accum == GRAD_ACCUM or micro == math.ceil(len(tr) / MICRO_BS):
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        update += 1; accum = 0
        el = time.time() - t0
        if update % 100 == 0:
            print(f'step {update}/{steps} loss {loss.item():.4f} {el/60:.1f} min, eta {el/update*(steps-update)/60:.1f} min', flush=True)
        if el > TRAIN_MINUTES * 60:
            print(f'time guard: stopping at step {update}/{steps}', flush=True); break
d = evaluate(dev); p = 1 / (1 + np.exp(-d)); yv = dev.label.values
print('dev logloss', round(float(-np.mean(yv*np.log(np.clip(p,1e-7,1)) + (1-yv)*np.log(np.clip(1-p,1e-7,1)))), 4),
      'acc', round(float(np.mean((p > 0.5) == yv)), 4), flush=True)


In [ ]:
# Score the same three groups as the Kaggle notebook. Chunk files make scoring resumable.
SCORE_BS = 32
CHUNK_ROWS = 20_000
chunk_dir = OUT / 'bge_chunks'
chunk_dir.mkdir(exist_ok=True)

for g in ('tune', 'holdout', 'test'):
    final_path = OUT / f'ce_{g}.parquet'
    if final_path.exists():
        print(g, 'already scored; keeping', final_path.name, flush=True)
        continue
    df = pd.read_parquet(BUNDLE / f'score_{g}.parquet')
    order = np.argsort((df.ta.str.len() + df.tb.str.len()).values, kind='stable')
    df = df.iloc[order].reset_index(drop=True)
    n_chunks = math.ceil(len(df) / CHUNK_ROWS)
    for k in range(n_chunks):
        part_path = chunk_dir / f'{g}_{k:05d}.parquet'
        if part_path.exists():
            continue
        lo, hi = k * CHUNK_ROWS, min((k + 1) * CHUNK_ROWS, len(df))
        part = df.iloc[lo:hi].reset_index(drop=True)
        ce = evaluate(part, bs=SCORE_BS)
        tmp = part_path.with_suffix('.tmp.parquet')
        pd.DataFrame({'lid': part.lid, 'rid': part.rid, 'ce': ce}).to_parquet(tmp, index=False)
        tmp.replace(part_path)
        print(f'{g}: scored chunk {k+1}/{n_chunks} ({hi}/{len(df)})', flush=True)
    parts = [pd.read_parquet(chunk_dir / f'{g}_{k:05d}.parquet') for k in range(n_chunks)]
    pd.concat(parts, ignore_index=True).to_parquet(final_path, index=False)
    print(g, len(df), 'wrote', final_path.name, flush=True)
print('DONE — download ce_tune.parquet, ce_holdout.parquet, ce_test.parquet from My Drive/ber_ce_big/out/.', flush=True)


If Colab disconnects during scoring, reconnect to a GPU and run the scoring cell again; completed chunks are skipped. If it disconnects during training, training restarts from the beginning, so allow the training cell to finish before a long idle period.